# Email Spam Detection System
Machine Learning Assignment

In [19]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from scipy.sparse import hstack

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()


# ----------------------------
# CLEAN TEXT
# ----------------------------
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' url ', text)
    text = re.sub(r'\S+@\S+', ' email ', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]

    return " ".join(words)


# ----------------------------
# ENGINEER FEATURES
# ----------------------------
def features(text):
    text = str(text).lower()

    return [
        int(bool(re.search(r'\d', text))),      # numbers
        int('%' in text or 'off' in text),      # discount
        int('free' in text),                    # free
        int('urgent' in text or 'hurry' in text),
        int('win' in text or 'winner' in text),
        len(text)
    ]


# ----------------------------
# LOAD DATA
# ----------------------------
df = pd.read_csv("spam.csv", encoding='latin-1')[['v1','v2']]
df.columns = ['label', 'text']
df['label'] = df['label'].map({'ham':0, 'spam':1})

df['clean'] = df['text'].apply(preprocess)

# ----------------------------
# TF-IDF (SPARSE - IMPORTANT)
# ----------------------------
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_tfidf = vectorizer.fit_transform(df['clean'])

# ----------------------------
# EXTRA FEATURES
# ----------------------------
X_extra = np.array([features(t) for t in df['text']])

# ----------------------------
# COMBINE (SPARSE SAFE)
# ----------------------------
X = hstack([X_tfidf, X_extra])
y = df['label']

# ----------------------------
# TRAIN TEST SPLIT
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ----------------------------
# MODEL (FAST + ACCURATE)
# ----------------------------
model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced'
)

model.fit(X_train, y_train)

# ----------------------------
# EVALUATION
# ----------------------------
pred = model.predict(X_test)

print(classification_report(y_test, pred))

# ----------------------------
# SAVE
# ----------------------------
joblib.dump(model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")
import numpy as np
import re
import joblib
from scipy.sparse import hstack

model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' url ', text)
    text = re.sub(r'\S+@\S+', ' email ', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text


def features(text):
    text = str(text).lower()

    return [
        int(bool(re.search(r'\d', text))),
        int('%' in text or 'off' in text),
        int('free' in text),
        int('urgent' in text or 'hurry' in text),
        int('win' in text or 'winner' in text),
        len(text)
    ]


def predict_email(text):
    clean = preprocess(text)

    tfidf = vectorizer.transform([clean])
    extra = np.array([features(text)])

    X = hstack([tfidf, extra])

    prob = model.predict_proba(X)[0][1]

    return {
        "label": "Spam" if prob > 0.5 else "Ham",
        "confidence": float(prob)
    }


# TEST
print(predict_email("First order? Get 50% off!"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


              precision    recall  f1-score   support

           0       0.99      0.96      0.97       966
           1       0.78      0.93      0.85       149

    accuracy                           0.96      1115
   macro avg       0.89      0.94      0.91      1115
weighted avg       0.96      0.96      0.96      1115

{'label': 'Spam', 'confidence': 0.5057878873390177}


## Preprocessing Function

In [2]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return " ".join(words)

## Load Dataset

In [3]:
df = pd.read_csv("spam.csv", encoding='latin-1')[['v1','v2']]
df.columns = ['label', 'text']
df['label'] = df['label'].map({'ham':0, 'spam':1})
df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


## Apply Preprocessing

In [4]:
df['clean_text'] = df['text'].apply(preprocess_text)

## Handle Imbalance

In [5]:
spam = df[df.label == 1]
ham = df[df.label == 0]

spam_upsampled = resample(spam,
                         replace=True,
                         n_samples=len(ham),
                         random_state=42)

df_balanced = pd.concat([ham, spam_upsampled])

## Feature Extraction (TF-IDF)

In [6]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df_balanced['clean_text'])
y = df_balanced['label']

## Train-Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Train Models

In [8]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear')
}

for name, model in models.items():
    model.fit(X_train, y_train)

## Evaluate Models

In [9]:
for name, model in models.items():
    y_pred = model.predict(X_test)

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))


Naive Bayes
Accuracy: 0.9715025906735751
Precision: 0.9715489989462592
Recall: 0.9705263157894737
F1 Score: 0.9710373880989994

Logistic Regression
Accuracy: 0.9870466321243523
Precision: 0.9914984059511158
Recall: 0.9821052631578947
F1 Score: 0.9867794817556849

SVM
Accuracy: 0.9958549222797928
Precision: 0.9968354430379747
Recall: 0.9947368421052631
F1 Score: 0.9957850368809273


## Save Best Model

In [10]:
best_model = models["Logistic Regression"]

joblib.dump(best_model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

['vectorizer.pkl']

## Prediction Function

In [11]:
def predict_email(text):
    cleaned = preprocess_text(text)
    vec = vectorizer.transform([cleaned])
    pred = best_model.predict(vec)[0]

    return "Spam" if pred == 1 else "Ham" 

## Test Prediction

In [16]:
predict_email("Use this code to log back into Instagram safely.576214 Meta © Instagram. Meta Platforms, Inc., 1601 Willow Road, Menlo Park, CA 94025")

'Ham'